In [ ]:
# @markdown # Select profile of Fooocus and start/restart
#
# Clean-output version: only stage banners + step labels are printed.
# All apt / pip / git / wget noise is captured and shown only if a step fails.
# Paste this whole cell into a Colab code cell and run.

import os
import subprocess

VENV = "/content/fooocus-venv"
VENV_PYTHON = f"{VENV}/bin/python"
FOOOCUS_DIR = "/content/Fooocus_extend"

# ---------- UI controls (keep exactly as-is so Colab renders the form) ----------
Fooocus_Profile = "realistic" #@param ["default", "realistic", "anime"]
Fooocus_Theme = "dark" #@param ["dark", "light"]
Tunnel = "gradio" #@param ["gradio", "cloudflared"]
Memory_patch = True #@param {type:"boolean"}
GoogleDrive_output = False #@param {type:"boolean"}

# ---------- helpers ----------
def _banner(title):
    print(f"\n{'=' * 64}\n  {title}\n{'=' * 64}")

def _step(msg):
    print(f"  · {msg}")

def _run(cmd, label=None, check=True):
    """Run a command silently. Print only a label; surface output on failure."""
    if label:
        _step(label)
    res = subprocess.run(
        cmd,
        shell=isinstance(cmd, str),
        capture_output=True,
        text=True,
        env={**os.environ, "DEBIAN_FRONTEND": "noninteractive"},
    )
    if res.returncode != 0:
        tail = (res.stderr or res.stdout or "")[-3000:]
        print(f"    ✗ failed (exit {res.returncode})")
        if tail:
            print(tail)
        if check:
            raise SystemExit(res.returncode)
    return res

# ---------- Stage 1: build Python 3.10 venv ----------
if not os.path.exists(VENV_PYTHON):
    _banner("Stage 1/4 · Building Python 3.10 virtual environment")
    _run("apt-get update -y", "apt: update package index")
    _run("apt-get install -y python3.10 python3.10-venv python3.10-distutils", "apt: install python3.10")
    _run(["python3.10", "-m", "venv", VENV], "venv: create Python 3.10 environment")
    _run([VENV_PYTHON, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], "pip: upgrade pip / setuptools / wheel")
    _run([VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "numpy==1.26.4", "cffi==1.17.0"], "pip: numpy 1.26.4 + cffi 1.17.0")
    _run([VENV_PYTHON, "-m", "pip", "install", "--no-cache-dir", "--no-deps", "pygit2==1.15.1"], "pip: pygit2 1.15.1")
    _run([VENV_PYTHON, "-m", "pip", "install", "torch==2.7.1+cu128", "torchvision==0.22.1+cu128", "torchaudio==2.7.1+cu128", "--index-url", "https://download.pytorch.org/whl/cu128"], "pip: torch 2.7.1 + CUDA 12.8")
    _run([VENV_PYTHON, "-m", "pip", "install", "matplotlib-inline==0.1.7"], "pip: matplotlib-inline")
    _run([VENV_PYTHON, "-m", "pip", "install", "IPython"], "pip: IPython")
else:
    _banner("Stage 1/4 · Virtual environment already present — skipping setup")

# ---------- Stage 2: verify runtime ----------
_banner("Stage 2/4 · Verifying runtime (Python / NumPy / Torch / CUDA)")
_check = _run([VENV_PYTHON, "-c", "import sys,numpy,torch;print('Python:',sys.executable);print('NumPy:',numpy.__version__,numpy.__file__);print('Torch:',torch.__version__);print('CUDA:',torch.cuda.is_available())"], label=None, check=False)
print(_check.stdout.rstrip())

# ---------- Stage 3: clone / update Fooocus_extend ----------
_banner("Stage 3/4 · Preparing Fooocus_extend")
if os.path.isdir(os.path.join(FOOOCUS_DIR, ".git")):
    _run(["git", "-C", FOOOCUS_DIR, "pull", "--ff-only"], "git: update existing checkout")
else:
    _run(["git", "clone", "https://github.com/shaitanzx/Fooocus_extend.git", FOOOCUS_DIR], "git: clone dev branch")

os.chdir(FOOOCUS_DIR)

if Tunnel == "cloudflared":
    _deb = "/content/cloudflared-linux-amd64.deb"
    _run(["wget", "-q", "-O", _deb, "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], "wget: cloudflared")
    _run(["dpkg", "-i", _deb], "dpkg: install cloudflared")
    _patcher = f"{FOOOCUS_DIR}/patcher_tunel.py"
    _run(["wget", "-q", "-O", _patcher, "https://github.com/shaitanzx/Fooocus_extend/raw/main/patcher_tunel.py"], "wget: patcher_tunel.py")
    _run([VENV_PYTHON, _patcher], "patch: apply tunnel patch")
    if os.path.exists(_patcher):
        os.remove(_patcher)

# ---------- optional: Google Drive output ----------
output_folder = ""
if GoogleDrive_output:
    _step("Mounting Google Drive…")
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    os.makedirs("/content/drive/MyDrive/outputs", exist_ok=True)
    output_folder = "--output-path /content/drive/MyDrive/outputs"

# ---------- build launch flags (no stray empty args) ----------
profile = {"default": "", "realistic": "--preset realistic", "anime": "--preset anime"}[Fooocus_Profile]
theme = "--theme dark" if Fooocus_Theme == "dark" else ""
patch = "--always-high-vram --all-in-fp16" if Memory_patch else ""
launch_args = " ".join(a for a in [profile, theme, "--share", patch, output_folder] if a)

# ---------- Stage 4: launch ----------
_banner("Stage 4/4 · Launching Fooocus_extend")
print(f"  profile : {profile or '(default)'}")
print(f"  theme   : {Fooocus_Theme}")
print(f"  tunnel  : {Tunnel}")
print(f"  vram    : {patch or '(default)'}")
print(f"  output  : {output_folder or '(local)'}")
print(" " + "─" * 63)
# The app itself streams live (Gradio URL, model loading) — that is the work stage.
!{VENV_PYTHON} entry_with_update.py {launch_args}